In [1]:
import os
import importlib.util
from dotenv import load_dotenv

# 프로젝트의 .env 파일 읽기
load_dotenv("/Users/anna/bulsae/.env")

# Neo4j 드라이버 설치 여부
if importlib.util.find_spec("neo4j") is not None:
    print("Neo4j 드라이버: 설치됨")
else:
    print("Neo4j 드라이버: 설치되지 않음")

# 접속 정보가 설정되어 있는지만 확인
for key in ["NEO4J_URI", "NEO4J_USERNAME", "NEO4J_PASSWORD"]:
    print(f"{key}:", "설정됨" if os.getenv(key) else "설정되지 않음")

Neo4j 드라이버: 설치됨
NEO4J_URI: 설정되지 않음
NEO4J_USERNAME: 설정되지 않음
NEO4J_PASSWORD: 설정되지 않음


In [2]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/anna/bulsae/.env", override=True)

for key in [
    "NEO4J_URI",
    "NEO4J_USERNAME",
    "NEO4J_PASSWORD",
    "NEO4J_DATABASE",
]:
    print(f"{key}:", "설정됨" if os.getenv(key) else "설정되지 않음")

NEO4J_URI: 설정됨
NEO4J_USERNAME: 설정됨
NEO4J_PASSWORD: 설정됨
NEO4J_DATABASE: 설정됨


In [3]:
import os
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(
        os.getenv("NEO4J_USERNAME"),
        os.getenv("NEO4J_PASSWORD"),
    ),
)

driver.verify_connectivity()

print("Neo4j Aura 연결 성공!")

Neo4j Aura 연결 성공!


In [4]:
database_name = os.getenv("NEO4J_DATABASE")

# 기존에 사용 중인 노드 라벨 확인
labels, _, _ = driver.execute_query(
    "CALL db.labels() YIELD label RETURN label ORDER BY label",
    database_=database_name,
)

print("=== 기존 노드 라벨 ===")
for record in labels:
    print(record["label"])

# 기존 인덱스 이름과 종류 확인
indexes, _, _ = driver.execute_query(
    """
    SHOW INDEXES
    YIELD name, type, labelsOrTypes
    RETURN name, type, labelsOrTypes
    ORDER BY name
    """,
    database_=database_name,
)

print("\n=== 기존 인덱스 ===")
for record in indexes:
    print(record["name"], "|", record["type"], "|", record["labelsOrTypes"])

=== 기존 노드 라벨 ===

=== 기존 인덱스 ===
index_1b9dcc97 | LOOKUP | None
index_460996c0 | LOOKUP | None


In [5]:
import json
import os
from pathlib import Path

chunk_path = Path("/Users/anna/bulsae/data/processed/news_chunks_v1.jsonl")

# 파일에서 첫 번째 청크 읽기
with chunk_path.open("r", encoding="utf-8") as file:
    first_chunk = json.loads(file.readline())

# 첫 번째 청크를 Neo4j에 저장
records, summary, _ = driver.execute_query(
    """
    MERGE (c:BulsaeChunk {chunk_id: $chunk_id})
    ON CREATE SET
        c.document_id = $document_id,
        c.chunk_index = $chunk_index,
        c.text = $text,
        c.title = $title,
        c.original_url = $original_url,
        c.published_at = $published_at,
        c.chunk_config_json = $chunk_config_json
    RETURN c.chunk_id AS chunk_id
    """,
    chunk_id=first_chunk["chunk_id"],
    document_id=first_chunk["document_id"],
    chunk_index=first_chunk["chunk_index"],
    text=first_chunk["text"],
    title=first_chunk["title"],
    original_url=first_chunk["original_url"],
    published_at=first_chunk["published_at"],
    chunk_config_json=json.dumps(
        first_chunk["chunk_config"], ensure_ascii=False
    ),
    database_=os.getenv("NEO4J_DATABASE"),
)

print("저장된 청크 ID:", records[0]["chunk_id"])
print("새로 만들어진 노드 수:", summary.counters.nodes_created)


저장된 청크 ID: NEWS-23178d84642bd6c9-84dd7bcc9ee5-0-484874aaf488
새로 만들어진 노드 수: 1


In [6]:
import json
import os
from pathlib import Path

chunk_path = Path("/Users/anna/bulsae/data/processed/news_chunks_v1.jsonl")
database_name = os.getenv("NEO4J_DATABASE")

# 청크 ID가 중복되지 않도록 제약 조건 만들기
driver.execute_query(
    """
    CREATE CONSTRAINT bulsae_chunk_id_unique IF NOT EXISTS
    FOR (c:BulsaeChunk)
    REQUIRE c.chunk_id IS UNIQUE
    """,
    database_=database_name,
)

# 청크 여러 개를 한 번에 저장하는 Cypher
query = """
UNWIND $rows AS row
MERGE (c:BulsaeChunk {chunk_id: row.chunk_id})
SET c.document_id = row.document_id,
    c.chunk_index = row.chunk_index,
    c.text = row.text,
    c.title = row.title,
    c.original_url = row.original_url,
    c.published_at = row.published_at,
    c.company_ids = row.company_ids,
    c.chunk_config_json = row.chunk_config_json
"""

batch = []
batch_size = 250
processed = 0

with chunk_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        chunk = json.loads(line)

        batch.append({
            "chunk_id": chunk["chunk_id"],
            "document_id": chunk["document_id"],
            "chunk_index": chunk["chunk_index"],
            "text": chunk["text"],
            "title": chunk["title"],
            "original_url": chunk["original_url"],
            "published_at": chunk["published_at"],
            "company_ids": chunk["company_ids"],
            "chunk_config_json": json.dumps(
                chunk["chunk_config"], ensure_ascii=False
            ),
        })

        if len(batch) == batch_size:
            driver.execute_query(query, rows=batch, database_=database_name)
            processed += len(batch)
            print(f"처리한 청크: {processed}개")
            batch = []

# 마지막으로 250개보다 적게 남은 청크 저장
if batch:
    driver.execute_query(query, rows=batch, database_=database_name)
    processed += len(batch)

print("청크 적재 완료!")
print("처리한 전체 청크:", processed, "개")

처리한 청크: 250개
처리한 청크: 500개
처리한 청크: 750개
처리한 청크: 1000개
처리한 청크: 1250개
처리한 청크: 1500개
처리한 청크: 1750개
처리한 청크: 2000개
처리한 청크: 2250개
처리한 청크: 2500개
처리한 청크: 2750개
처리한 청크: 3000개
처리한 청크: 3250개
처리한 청크: 3500개
처리한 청크: 3750개
처리한 청크: 4000개
처리한 청크: 4250개
처리한 청크: 4500개
처리한 청크: 4750개
처리한 청크: 5000개
처리한 청크: 5250개
처리한 청크: 5500개
처리한 청크: 5750개
처리한 청크: 6000개
처리한 청크: 6250개
처리한 청크: 6500개
처리한 청크: 6750개
처리한 청크: 7000개
처리한 청크: 7250개
처리한 청크: 7500개
처리한 청크: 7750개
처리한 청크: 8000개
처리한 청크: 8250개
처리한 청크: 8500개
처리한 청크: 8750개
처리한 청크: 9000개
처리한 청크: 9250개
처리한 청크: 9500개
처리한 청크: 9750개
처리한 청크: 10000개
처리한 청크: 10250개
처리한 청크: 10500개
처리한 청크: 10750개
처리한 청크: 11000개
처리한 청크: 11250개
처리한 청크: 11500개
처리한 청크: 11750개
처리한 청크: 12000개
처리한 청크: 12250개
청크 적재 완료!
처리한 전체 청크: 12309 개


In [7]:
import json

# 청크 파일에 들어 있는 ID 목록 읽기
with chunk_path.open("r", encoding="utf-8") as file:
    chunk_ids = [
        json.loads(line)["chunk_id"]
        for line in file
        if line.strip()
    ]

# 파일의 ID 중 Neo4j에서 찾을 수 있는 청크 수 확인
records, _, _ = driver.execute_query(
    """
    UNWIND $chunk_ids AS chunk_id
    OPTIONAL MATCH (c:BulsaeChunk {chunk_id: chunk_id})
    RETURN count(c) AS found
    """,
    chunk_ids=chunk_ids,
    database_=database_name,
)

found = records[0]["found"]

print("파일의 청크 수:", len(chunk_ids))
print("Neo4j에서 찾은 청크 수:", found)
print("찾지 못한 청크 수:", len(chunk_ids) - found)

파일의 청크 수: 12309
Neo4j에서 찾은 청크 수: 12309
찾지 못한 청크 수: 0


In [8]:
import json
import os
from pathlib import Path

news_path = Path("/Users/anna/bulsae/data/collected/news.jsonl")

# 원본 뉴스 파일에서 첫 번째 기사 읽기
with news_path.open("r", encoding="utf-8") as file:
    first_news = json.loads(file.readline())

# 뉴스 원문 한 건 저장
records, summary, _ = driver.execute_query(
    """
    MERGE (d:BulsaeDocument {document_id: $document_id})
    ON CREATE SET
        d.title = $title,
        d.content = $content,
        d.original_url = $original_url,
        d.published_at = $published_at,
        d.company_ids = $company_ids
    RETURN d.document_id AS document_id
    """,
    document_id=first_news["document_id"],
    title=first_news["title"],
    content=first_news["content"],
    original_url=first_news["original_url"],
    published_at=first_news["published_at"],
    company_ids=first_news["company_ids"],
    database_=os.getenv("NEO4J_DATABASE"),
)

print("저장된 기사 ID:", records[0]["document_id"])
print("새로 만들어진 문서 노드 수:", summary.counters.nodes_created)

저장된 기사 ID: NEWS-23178d84642bd6c9
새로 만들어진 문서 노드 수: 1


In [9]:
import json
import os
from pathlib import Path

news_path = Path("/Users/anna/bulsae/data/collected/news.jsonl")
database_name = os.getenv("NEO4J_DATABASE")

# 문서 ID가 중복되지 않도록 제약 조건 만들기
driver.execute_query(
    """
    CREATE CONSTRAINT bulsae_document_id_unique IF NOT EXISTS
    FOR (d:BulsaeDocument)
    REQUIRE d.document_id IS UNIQUE
    """,
    database_=database_name,
)

# 뉴스 원문 여러 건을 저장하는 Cypher
query = """
UNWIND $rows AS row
MERGE (d:BulsaeDocument {document_id: row.document_id})
SET d.title = row.title,
    d.content = row.content,
    d.original_url = row.original_url,
    d.published_at = row.published_at,
    d.company_ids = row.company_ids
"""

batch = []
batch_size = 100
processed = 0

with news_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        article = json.loads(line)

        batch.append({
            "document_id": article["document_id"],
            "title": article["title"],
            "content": article["content"],
            "original_url": article["original_url"],
            "published_at": article["published_at"],
            "company_ids": article["company_ids"],
        })

        if len(batch) == batch_size:
            driver.execute_query(query, rows=batch, database_=database_name)
            processed += len(batch)
            print(f"처리한 기사: {processed}건")
            batch = []

# 마지막으로 100건보다 적게 남은 기사 저장
if batch:
    driver.execute_query(query, rows=batch, database_=database_name)
    processed += len(batch)

print("뉴스 원문 적재 완료!")
print("처리한 전체 기사:", processed, "건")

처리한 기사: 100건
처리한 기사: 200건
처리한 기사: 300건
처리한 기사: 400건
처리한 기사: 500건
처리한 기사: 600건
처리한 기사: 700건
처리한 기사: 800건
처리한 기사: 900건
처리한 기사: 1000건
처리한 기사: 1100건
처리한 기사: 1200건
처리한 기사: 1300건
처리한 기사: 1400건
처리한 기사: 1500건
처리한 기사: 1600건
처리한 기사: 1700건
처리한 기사: 1800건
처리한 기사: 1900건
처리한 기사: 2000건
처리한 기사: 2100건
처리한 기사: 2200건
처리한 기사: 2300건
처리한 기사: 2400건
처리한 기사: 2500건
처리한 기사: 2600건
처리한 기사: 2700건
처리한 기사: 2800건
처리한 기사: 2900건
처리한 기사: 3000건
처리한 기사: 3100건
처리한 기사: 3200건
처리한 기사: 3300건
처리한 기사: 3400건
처리한 기사: 3500건
처리한 기사: 3600건
처리한 기사: 3700건
처리한 기사: 3800건
처리한 기사: 3900건
처리한 기사: 4000건
처리한 기사: 4100건
처리한 기사: 4200건
처리한 기사: 4300건
처리한 기사: 4400건
처리한 기사: 4500건
뉴스 원문 적재 완료!
처리한 전체 기사: 4580 건


In [10]:
# 저장된 원문 문서 수 확인
documents, _, _ = driver.execute_query(
    """
    MATCH (d:BulsaeDocument)
    RETURN count(d) AS document_count
    """,
    database_=database_name,
)

# 각 청크의 document_id에 해당하는 원문이 있는지 확인
chunks, _, _ = driver.execute_query(
    """
    MATCH (c:BulsaeChunk)
    OPTIONAL MATCH (d:BulsaeDocument {document_id: c.document_id})
    RETURN count(c) AS chunk_count,
           count(d) AS matched_count
    """,
    database_=database_name,
)

document_count = documents[0]["document_count"]
chunk_count = chunks[0]["chunk_count"]
matched_count = chunks[0]["matched_count"]

print("저장된 원문 문서 수:", document_count)
print("저장된 청크 수:", chunk_count)
print("원문을 찾은 청크 수:", matched_count)
print("원문을 찾지 못한 청크 수:", chunk_count - matched_count)

저장된 원문 문서 수: 4580
저장된 청크 수: 12309
원문을 찾은 청크 수: 12309
원문을 찾지 못한 청크 수: 0


In [11]:
import os
import importlib.util

if importlib.util.find_spec("openai") is not None:
    print("OpenAI 패키지: 설치됨")
else:
    print("OpenAI 패키지: 설치되지 않음")

if os.getenv("OPENAI_API_KEY"):
    print("OpenAI API 키: 설정됨")
else:
    print("OpenAI API 키: 설정되지 않음")

OpenAI 패키지: 설치되지 않음
OpenAI API 키: 설정되지 않음


In [12]:
import openai

print("OpenAI 패키지 불러오기 성공!")

OpenAI 패키지 불러오기 성공!


In [13]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/anna/bulsae/.env", override=True)

print("API 키:", "설정됨" if os.getenv("OPENAI_API_KEY") else "설정되지 않음")

API 키: 설정됨


In [14]:
import json
from pathlib import Path
from openai import OpenAI

# 저장된 청크 파일에서 첫 번째 청크 읽기
chunk_path = Path("/Users/anna/bulsae/data/processed/news_chunks_v1.jsonl")

with chunk_path.open("r", encoding="utf-8") as file:
    first_chunk = json.loads(file.readline())

# OpenAI 클라이언트 준비
client = OpenAI()

# 청크 본문을 임베딩으로 변환
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=first_chunk["text"],
)

embedding = response.data[0].embedding

print("청크 ID:", first_chunk["chunk_id"])
print("벡터의 숫자 개수:", len(embedding))
print("벡터 앞의 숫자 5개:", embedding[:5])

청크 ID: NEWS-23178d84642bd6c9-84dd7bcc9ee5-0-484874aaf488
벡터의 숫자 개수: 1536
벡터 앞의 숫자 5개: [0.03289794921875, 0.01338958740234375, -0.001056671142578125, 0.0246734619140625, 0.0103912353515625]


In [16]:
import os

records, _, _ = driver.execute_query(
    """
    MATCH (c:BulsaeChunk {chunk_id: $chunk_id})
    WHERE c.text = $text
    SET c.embedding = $embedding,
        c.embedding_model = $model
    RETURN c.chunk_id AS chunk_id,
           size(c.embedding) AS dimensions,
           c.embedding_model AS model
    """,
    chunk_id=first_chunk["chunk_id"],
    text=first_chunk["text"],
    embedding=embedding,
    model="text-embedding-3-small",
    database_=os.getenv("NEO4J_DATABASE"),
)

if records:
    print("저장된 청크 ID:", records[0]["chunk_id"])
    print("저장된 벡터 숫자 개수:", records[0]["dimensions"])
    print("임베딩 모델:", records[0]["model"])
else:
    print("청크 ID와 본문이 일치하는 노드를 찾지 못했어요.")

저장된 청크 ID: NEWS-23178d84642bd6c9-84dd7bcc9ee5-0-484874aaf488
저장된 벡터 숫자 개수: 1536
임베딩 모델: text-embedding-3-small


In [17]:
import os
from openai import OpenAI

client = OpenAI()
database_name = os.getenv("NEO4J_DATABASE")

# 1. 아직 임베딩이 없는 청크 5개 가져오기
records, _, _ = driver.execute_query(
    """
    MATCH (c:BulsaeChunk)
    WHERE c.embedding IS NULL
    RETURN c.chunk_id AS chunk_id, c.text AS text
    ORDER BY c.chunk_id
    LIMIT 5
    """,
    database_=database_name,
)

if not records:
    print("임베딩이 필요한 청크가 없어요.")
else:
    # 2. 청크 본문 5개를 한 번에 임베딩
    texts = [record["text"] for record in records]

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts,
    )

    # 3. 각 청크 ID와 임베딩 결과를 묶기
    rows = [
        {
            "chunk_id": records[item.index]["chunk_id"],
            "text": records[item.index]["text"],
            "embedding": item.embedding,
        }
        for item in response.data
    ]

    # 4. 같은 청크에 벡터 저장하기
    saved, _, _ = driver.execute_query(
        """
        UNWIND $rows AS row
        MATCH (c:BulsaeChunk {chunk_id: row.chunk_id})
        WHERE c.text = row.text AND c.embedding IS NULL
        SET c.embedding = row.embedding,
            c.embedding_model = 'text-embedding-3-small'
        RETURN count(c) AS saved_count
        """,
        rows=rows,
        database_=database_name,
    )

    print("요청한 청크 수:", len(records))
    print("저장한 임베딩 수:", saved[0]["saved_count"])

요청한 청크 수: 5
저장한 임베딩 수: 5


In [18]:
records, _, _ = driver.execute_query(
    """
    MATCH (c:BulsaeChunk)
    RETURN
        count(c) AS total,
        count(c.embedding) AS with_embedding,
        count(c) - count(c.embedding) AS without_embedding
    """,
    database_=database_name,
)

status = records[0]

print("전체 청크:", status["total"], "개")
print("임베딩이 있는 청크:", status["with_embedding"], "개")
print("임베딩이 없는 청크:", status["without_embedding"], "개")

전체 청크: 12309 개
임베딩이 있는 청크: 6 개
임베딩이 없는 청크: 12303 개


In [19]:
records, _, _ = driver.execute_query(
    """
    MATCH (c:BulsaeChunk)
    RETURN count(c) AS total,
           count(c.embedding) AS with_embedding,
           count(c) - count(c.embedding) AS without_embedding
    """,
    database_=database_name,
)

status = records[0]

print("전체 청크:", status["total"])
print("임베딩 완료:", status["with_embedding"])
print("임베딩 필요:", status["without_embedding"])

전체 청크: 12309
임베딩 완료: 6
임베딩 필요: 12303


In [ ]:
import json
import os
from pathlib import Path
from openai import OpenAI

client = OpenAI()
database_name = os.getenv("NEO4J_DATABASE")

chunk_path = Path("/Users/anna/bulsae/data/processed/news_chunks_v1.jsonl")

batch_size = 50
processed = 0
used_tokens = 0


def embed_batch(chunk_ids):
    # 이 묶음에서 아직 임베딩이 없는 청크 조회
    records, _, _ = driver.execute_query(
        """
        UNWIND $chunk_ids AS chunk_id
        MATCH (c:BulsaeChunk {chunk_id: chunk_id})
        WHERE c.embedding IS NULL
        RETURN c.chunk_id AS chunk_id, c.text AS text
        ORDER BY c.chunk_id
        """,
        chunk_ids=chunk_ids,
        database_=database_name,
    )

    if not records:
        return 0, 0

    # 청크 본문을 임베딩으로 변환
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[record["text"] for record in records],
    )

    rows = [
        {
            "chunk_id": records[item.index]["chunk_id"],
            "text": records[item.index]["text"],
            "embedding": item.embedding,
        }
        for item in response.data
    ]

    if len(rows) != len(records):
        raise ValueError("요청한 청크 수와 임베딩 결과 수가 달라요.")

    # 해당 청크에 임베딩 저장
    saved, _, _ = driver.execute_query(
        """
        UNWIND $rows AS row
        MATCH (c:BulsaeChunk {chunk_id: row.chunk_id})
        WHERE c.text = row.text AND c.embedding IS NULL
        SET c.embedding = row.embedding,
            c.embedding_model = 'text-embedding-3-small'
        RETURN count(c) AS saved_count
        """,
        rows=rows,
        database_=database_name,
    )

    saved_count = saved[0]["saved_count"]

    if saved_count != len(records):
        raise ValueError(
            "일부 청크가 변경됐거나 다른 작업에서 먼저 처리됐어요. "
            "저장 상태를 확인해주세요."
        )

    return saved_count, response.usage.total_tokens


# 파일의 청크 ID를 50개씩 묶어 처리
batch = []

with chunk_path.open("r", encoding="utf-8") as file:
    for line in file:
        if not line.strip():
            continue

        chunk = json.loads(line)
        batch.append(chunk["chunk_id"])

        if len(batch) == batch_size:
            saved_count, tokens = embed_batch(batch)
            processed += saved_count
            used_tokens += tokens
            batch = []

            if processed and processed % 500 < batch_size:
                print(f"이번 실행에서 저장한 임베딩: {processed}개")

# 마지막으로 남은 청크 처리
if batch:
    saved_count, tokens = embed_batch(batch)
    processed += saved_count
    used_tokens += tokens

print("임베딩 작업 완료!")
print("이번 실행에서 저장한 임베딩:", processed, "개")
print("이번 실행에서 사용한 토큰:", used_tokens)

이번 실행에서 저장한 임베딩: 49개
이번 실행에서 저장한 임베딩: 549개
이번 실행에서 저장한 임베딩: 1049개
이번 실행에서 저장한 임베딩: 1549개
이번 실행에서 저장한 임베딩: 2049개
이번 실행에서 저장한 임베딩: 2549개
이번 실행에서 저장한 임베딩: 3049개
이번 실행에서 저장한 임베딩: 3544개
이번 실행에서 저장한 임베딩: 4044개
이번 실행에서 저장한 임베딩: 4544개
이번 실행에서 저장한 임베딩: 5044개
이번 실행에서 저장한 임베딩: 5544개
이번 실행에서 저장한 임베딩: 6044개
이번 실행에서 저장한 임베딩: 6544개
이번 실행에서 저장한 임베딩: 7044개
이번 실행에서 저장한 임베딩: 7544개
이번 실행에서 저장한 임베딩: 8044개
이번 실행에서 저장한 임베딩: 8544개
이번 실행에서 저장한 임베딩: 9044개
이번 실행에서 저장한 임베딩: 9544개
이번 실행에서 저장한 임베딩: 10044개
이번 실행에서 저장한 임베딩: 10544개
